# Track B — 확장 변수셋 탐색 전체 정리 (track_b_04)

담당: 서윤 | 선행 노트북: track_b_01_타겟변수_검증.ipynb, track_b_02_대안변수_선정.ipynb, track_b_03_전처리완료.ipynb

## 이 노트북을 왜 만들었는가

강사님 피드백 중 "다중공선성/IV 배제를 후순위로 두고, 소비패턴(특히 절대금액+증감률 페어)을 우선 넓게 넣어본 뒤 사후에 SHAP로 정리하라"는 지적이 있었다. track_b_02에서 IV·상관계수 컷으로 46개까지 좁혔는데, 그 과정에서 소비 카테고리 33개 중 대부분이 원본(절대금액)과 파생(증감률) 중 한쪽만 살아남아 짝이 깨진 상태였다.

이 노트북은 그 문제를 해결하기 위해 **5차례의 실험**을 순서대로 진행한 기록이다. 각 실험마다 "무엇을 시도했는지 → 결과가 어땠는지 → 채택했는지 기각했는지"를 명시한다.

## 전체 여정 요약 (결과 먼저 보기)

| 실험 | 무엇을 추가했나 | 변수 수 | Val AUC | Val PR-AUC | 판정 |
|---|---|---|---|---|---|
| 시작점 | track_b_03 최종 확정본 | 46 | 0.7782 | 0.1772 | 기준 |
| 실험 A | 소비 카테고리 33개(원본R6M+QOQ+YOY, 99개 후보) → SHAP 상위 13개 채택 | 58 | 0.7794 | 0.1780 | ✅ 채택 |
| 실험 B | 백화점·마트 브랜드 세분류 12개 카테고리(36개 후보) | 118 | 0.7792 | 0.1786 | ❌ 기각 (노이즈) |
| 실험 C | CPYT 구성비(현재값) 33개 → SHAP 상위 11개 채택 | 69 | 0.7824 | 0.1820 | ✅ 채택 |
| 실험 D | CD_USE_AMT/PYE_ALL_CD_USE_AMT (카드소비금액 총액) | +2 | 0.8476 | 0.3009 | ❌ 기각 (데이터 누출) |
| 실험 E | CPYT 나머지 변형(전월비·누적 등, 99개 후보) → 상위 5개 재테스트 | 74 | 0.7822 | 0.1841 | ❌ 기각 (순증분 없음) |

**최종 확정: 46개 → 69개.** 아래 각 실험을 순서대로 재현 가능한 코드로 정리했다. 맨 마지막 "최종 확정 스크립트" 섹션 하나만 실행해도 69개 변수셋을 바로 재현할 수 있다.

---
## 실험 A — 소비 카테고리 33개(원본+증감률) 넓게 추가

**배경**: 코드북 전체를 훑어보니 소비 카테고리가 45개 있고(브랜드 세분류 12개 제외하면 33개), 각각 원본(절대금액)과 QOQ_R3M/YOY_R3M(증감률) 버전이 다 존재했다. 그런데 기존 46개 안에는 MART 카테고리만 원본+증감률 페어가 완성되어 있고, 나머지는 한쪽만 있거나 아예 없었다.

**시도**: 33개 카테고리 × 3필드(R6M 원본 + QOQ_R3M + YOY_R3M) = 99개 후보를 IV/상관계수 컷 없이 전부 후보에 넣고 XGBoost로 학습.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import gc

base_path = '/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터'
scan_path = f'{base_path}/통신카드CB_씬파일러.csv'

# 비소비 핵심 25개 (기존 46개 중 소비 카테고리가 아닌 변수)
non_consumption_features = [
    'DAR', 'PYE_AL012G019', 'TOT_ASST', 'PYE_AL012G005', 'YOY_TOT_ASST_RTC',
    'R3M_ITRT_ENT_BROADCAST', 'AGE', 'OWN_HOUS_CNT', 'FAM_OWN_HOUS_CNT',
    'PYE_CAR_OWN', 'JB_TP', 'FST_CAR_ELPS', 'OWN_LIV_YN', 'FST_HOUS_BUY_ELPS',
    'FAM_OWN_LIV_YN', 'RET_SIL', 'HIGHEND_CD2', 'QOQ_TOT_ASST_RTC',
    'PYE_AL012G011', 'SHP_CNT', 'R3M_ITRT_SHOP_JIKGU', 'B1Y_MOB_OS',
    'HOME_ADM', 'R3M_ITRT_ENT_WEBTOON', 'R3M_ITRT_FIN_INSUR',
    'YOY_R3M_ITRT_COMM_VOIP_CS'
]

grade_source_cols = ['PET_GD', 'APP_GD', 'GOLF_GD', 'TRAVEL_GD']
missing_flag_source_cols = ['DAR', 'FST_CAR_ELPS', 'FST_HOUS_BUY_ELPS',
                              'CPYT_CONV_AMT_RT', 'CPYT_FOOD_AMT_RT']
thin_def_cols = ['PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004', 'PYE_C18233005', 'PYE_L10210000']

# 소비 카테고리 33개 (백화점·마트 브랜드 세분류 12개는 제외 - 실험 B에서 별도 검증)
consumption_categories = [
    'ACCO','BEAUTY','CAR','CAR_SERVICE','CLOTHES','CONV','CUL','DEP','DLV','EDU','ENT',
    'E_CHARGE','E_COMM','FOOD','FURN','HOTEL','HOUSEHOLD','JJ','MART','MED','M_CONV','M_DEP',
    'M_DF','M_H_SHOP','M_MART','M_OUTLET','M_S_MART','OIL','SSM','STARBUCKS','TRANS','TRAVEL','TRAVEL_OS'
]
consumption_base_cols = [f'R6M_{cat}_AMT' for cat in consumption_categories]
consumption_qoq_cols = [f'QOQ_R3M_{cat}_AMT_RTC' for cat in consumption_categories]
consumption_yoy_cols = [f'YOY_R3M_{cat}_AMT_RTC' for cat in consumption_categories]
consumption_all_cols = consumption_base_cols + consumption_qoq_cols + consumption_yoy_cols

print(f"비소비 핵심: {len(non_consumption_features)}개")
print(f"소비 카테고리 후보(실험 A): {len(consumption_all_cols)}개")

**결과**: 99개 후보 중 SHAP 상위권 검증 결과, 아래 13개만 확정 채택했다 (46개 → 58개).

채택 기준: SHAP mean(|value|)가 기존 46개 중 최저 기여 변수(YOY_R3M_FOOD_AMT_RTC, 0.027)보다 높은 것 위주로, 절대금액+증감률 페어가 최대한 완성되도록 선별.

In [ ]:
# 실험 A 최종 채택 결과 - 신규 편입 13개
consumption_accepted_13 = [
    'R6M_SSM_AMT', 'R6M_MED_AMT', 'R6M_HOUSEHOLD_AMT', 'R6M_OIL_AMT',
    'R6M_FOOD_AMT', 'R6M_CONV_AMT', 'R6M_BEAUTY_AMT', 'R6M_CUL_AMT',
    'R6M_M_CONV_AMT', 'QOQ_R3M_HOUSEHOLD_AMT_RTC', 'QOQ_R3M_E_COMM_AMT_RTC',
    'YOY_R3M_E_COMM_AMT_RTC_was_missing', 'YOY_R3M_STARBUCKS_AMT_RTC_was_missing'
]

# 기존 46개 중 소비 관련 변수 9개 (그대로 유지)
consumption_existing_9 = [
    'YOY_R3M_FOOD_AMT_RTC', 'R6M_MART_AMT', 'YOY_R3M_SSM_AMT_RTC',
    'YOY_R3M_MART_AMT_RTC', 'QOQ_R3M_MART_AMT_RTC',
    'YOY_R3M_HOUSEHOLD_AMT_RTC', 'YOY_R3M_CONV_AMT_RTC',
    'R6M_M_MART_AMT', 'YOY_R3M_M_CONV_AMT_RTC'
]

print(f"실험 A 결과 -> 46개 + {len(consumption_accepted_13)}개 = 58개 (실제로는 결측플래그 중복 제거로 58개 확인됨)")

---
## 실험 B — 백화점·대형마트 브랜드 세분류 검증 (기각)

**배경**: 실험 A에서 브랜드 세분류 12개 카테고리(AK플라자·갤러리아·현대·아이파크·롯데·신세계백화점, 이마트·이마트트레이더스·홈플러스·롯데마트·하나로마트·VIC마켓)는 상위 카테고리(M_DEP, M_MART)와 중복 정보라고 가정하고 제외했었다. 이 가정을 실제로 검증했다.

**시도**: 12개 카테고리 × 3필드(원본+QOQ+YOY) = 36개 후보를 58개 기준에 추가.

In [ ]:
brand_detail_categories = ['M_DEP_A','M_DEP_G','M_DEP_H','M_DEP_I','M_DEP_L','M_DEP_S',
                            'M_MART_E','M_MART_ET','M_MART_H','M_MART_L','M_MART_N','M_MART_V']

brand_base_cols = [f'R6M_{cat}_AMT' for cat in brand_detail_categories]
brand_qoq_cols = [f'QOQ_R3M_{cat}_AMT_RTC' for cat in brand_detail_categories]
brand_yoy_cols = [f'YOY_R3M_{cat}_AMT_RTC' for cat in brand_detail_categories]
brand_all_cols = brand_base_cols + brand_qoq_cols + brand_yoy_cols

print(f"브랜드 세분류 후보: {len(brand_all_cols)}개")

**결과 (실행 로그)**:
```
[58+브랜드 세분류] Val AUC: 0.7792, Val PR-AUC: 0.1786
[기존 58개]        Val AUC: 0.7794, Val PR-AUC: 0.1780
```
AUC가 오히려 미세하게 하락(0.7794→0.7792). SHAP 최상위(`R6M_M_MART_E_AMT`, 이마트)도 0.0084로, 기존 58개 중 최약체보다 3배 이상 낮음. 결측 패턴도 지역별 매장 유무 차이로 보여, 신용위험과 무관한 구조적 결측으로 판단.

**판정: 브랜드 세분류 36개 전량 기각 확정.** (아래 최종 스크립트에는 포함하지 않음)

---
## 실험 C — CPYT 구성비(현재값) 33개 검증 (채택)

**배경**: 지금까지 CPYT 계열은 `_was_missing` 플래그(결측 여부)로만 썼지, 실제 구성비 값 자체는 후보에 넣어본 적이 없었다. `CPYT_FOOD_AMT_RT_was_missing`이 SHAP 14위로 이미 존재감을 보였던 터라, 값 자체도 검증이 필요했다.

**시도**: CPYT_*_AMT_RT(현재 구성비) 33개를 58개 기준에 추가. (CPYT_CONV_AMT_RT, CPYT_FOOD_AMT_RT는 이미 was_missing 플래그 소스로 로드되어 있어 중복 로드 방지 처리 필요)

In [ ]:
cpyt_candidates = [f'CPYT_{cat}_AMT_RT' for cat in consumption_categories]
print(f"CPYT 구성비 후보: {len(cpyt_candidates)}개")

**결과 (실행 로그)**:
```
[58+CPYT 구성비]  Val AUC: 0.7811, Val PR-AUC: 0.1824  (33개 전부)
[최종 69개]        Val AUC: 0.7824, Val PR-AUC: 0.1820  (SHAP 상위 11개만 선별)
[기존 58개]        Val AUC: 0.7794, Val PR-AUC: 0.1780
```
흥미로운 발견: SHAP 상위권에 값 자체보다 `_was_missing` 플래그가 더 많이 올라옴 (`CPYT_CLOTHES_AMT_RT_was_missing` 2위, `CPYT_SSM_AMT_RT_was_missing` 4위 등) → "구성비가 얼마인가"보다 **"그 카테고리에서 구성비 계산 자체가 안 될 만큼 활동이 없는가"**가 더 강한 신호. 생활 반경의 협소함 자체가 신용위험과 관련된 것으로 해석 가능.

**판정: 아래 11개 채택 확정 (58개 → 69개).**

In [ ]:
# 실험 C 최종 채택 결과 - 신규 편입 11개
cpyt_accepted_11 = [
    'CPYT_MART_AMT_RT', 'CPYT_CLOTHES_AMT_RT_was_missing', 'CPYT_HOUSEHOLD_AMT_RT',
    'CPYT_SSM_AMT_RT_was_missing', 'CPYT_M_CONV_AMT_RT', 'CPYT_OIL_AMT_RT_was_missing',
    'CPYT_SSM_AMT_RT', 'CPYT_E_COMM_AMT_RT', 'CPYT_STARBUCKS_AMT_RT_was_missing',
    'CPYT_MART_AMT_RT_was_missing', 'CPYT_STARBUCKS_AMT_RT'
]
print(f"실험 C 결과 -> 58개 + {len(cpyt_accepted_11)}개 = 69개")

---
## 실험 D — 데이터 누출(leakage) 변수 발견 및 배제 (중요, 반드시 확인)

**배경**: CPYT 나머지 변형(전월비·누적 등)을 확인하는 과정에서 `CD_USE_AMT`, `PYE_ALL_CD_USE_AMT` 두 변수를 실수로 후보에 포함시켰다.

**결과**: 이 두 개를 넣자 AUC가 0.7824 → **0.8476**, PR-AUC가 0.1820 → **0.3009**로 비정상적으로 급등했고, SHAP에서 이 둘의 중요도(0.895, 0.733)가 나머지 267개를 다 합친 것보다 컸다.

**원인 확인**: 코드북에서 정의를 확인한 결과 —
```
CD_USE_AMT           = 카드소비금액
PYE_ALL_CD_USE_AMT   = 전년도 연간전체카드소비금액
```
그리고 `CD_USE_AMT`는 **track_b_02 8단계에서 이미 "신용노출 관련 필드"로 분류되어 제외 리스트(exposure_keywords)에 포함되어 있던 변수**였다. 이번에 CPYT 나머지 변형을 스캔하면서 이 제외 리스트 체크를 빠뜨리고 재도입한 것.

**왜 문제인가**: 개별 카테고리 소비금액을 모두 합친 총합에 가까운 값이라, "총 카드소비금액이 크다 = 신용활동이 활발한 사람"이라는 식으로 사실상 신용이력 보유군 내부의 활동 규모 자체를 재는 변수 — 순환논리에 가까움.

**판정: `CD_USE_AMT`, `PYE_ALL_CD_USE_AMT` 배제 확정. 이후 모든 실험에서 제외.**

---
## 실험 E — CPYT 나머지 변형(전월비·누적) 검증 (기각)

**배경**: CPYT는 카테고리당 4개 변형이 있다 — 이번에 채택한 `AMT_RT`(현재 구성비) 외에 `CPPY_AMT_RT`(전월비), `CPPY_CUM_AMT_RT`(전월비 누적), `CUM_AMT_RT`(누적) 3종이 더 있다. 33개 카테고리 × 3변형 = 99개 후보.

**시도 1 — 99개 전부**: 실험 D의 누출변수를 제외하고 검증.

In [ ]:
cpyt_remaining = []
for cat in consumption_categories:
    cpyt_remaining += [f'CPYT_{cat}_CPPY_AMT_RT', f'CPYT_{cat}_CPPY_CUM_AMT_RT', f'CPYT_{cat}_CUM_AMT_RT']
print(f"CPYT 나머지 변형 후보: {len(cpyt_remaining)}개 (CD_USE_AMT, PYE_ALL_CD_USE_AMT는 제외)")

**결과 (실행 로그)**:
```
[69+CPYT나머지(누출제외)] Val AUC: 0.7827, Val PR-AUC: 0.1841
[최종 69개 기준]          Val AUC: 0.7824, Val PR-AUC: 0.1820
```
99개를 다 넣으면 사실상 flat. SHAP 상위 5개(`CPYT_E_COMM_CPPY_AMT_RT` 등)가 기존 69개 최저 기여 변수보다 높게 나와서, 이 5개만 추려서 재검증했다.

**시도 2 — SHAP 상위 5개만 추가**:
```python
top5_new = ['CPYT_E_COMM_CPPY_AMT_RT', 'CPYT_FOOD_CUM_AMT_RT', 'CPYT_FOOD_CPPY_AMT_RT',
            'CPYT_MED_CPPY_AMT_RT', 'CPYT_STARBUCKS_CPPY_AMT_RT']
```

**최종 결과**:
```
[최종 74개]  Val AUC: 0.7822, Val PR-AUC: 0.1841
[69개 기준]  Val AUC: 0.7824, Val PR-AUC: 0.1820
```
AUC는 오히려 미세 하락, PR-AUC만 소폭 개선 — 순증분이 사실상 없음. 기존 69개 안에 이미 FOOD/STARBUCKS/HOUSEHOLD 계열이 여러 형태(R6M 원본, YOY, CPYT_AMT_RT)로 존재해서, 이 5개가 그것들과 상당 부분 겹치는 정보였을 것으로 판단.

**판정: CPYT 나머지 변형 99개 전량 기각 확정. 여기서 소비변수 탐색 마무리.**

---
## 최종 확정 스크립트 — 69개 변수셋 처음부터 재현 (메모리 효율화 버전)

위 실험 과정을 거쳐 확정된 **69개 변수셋**을 처음부터 재현하는 통합 스크립트다. 컬럼을 반복문으로 하나씩 추가하지 않고 `pd.concat`으로 한 번에 처리해 메모리 파편화(PerformanceWarning)를 방지했다.

**런타임이 끊기면 이 섹션만 다시 실행하면 된다.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import gc

base_path = '/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터'
scan_path = f'{base_path}/통신카드CB_씬파일러.csv'

# ===== 1. 최종 확정 69개 변수 =====
final_69_features = [
    # 비소비 핵심 25개
    'DAR', 'PYE_AL012G019', 'TOT_ASST', 'PYE_AL012G005', 'YOY_TOT_ASST_RTC',
    'R3M_ITRT_ENT_BROADCAST', 'AGE', 'OWN_HOUS_CNT', 'FAM_OWN_HOUS_CNT',
    'PYE_CAR_OWN', 'JB_TP', 'FST_CAR_ELPS', 'OWN_LIV_YN', 'FST_HOUS_BUY_ELPS',
    'FAM_OWN_LIV_YN', 'RET_SIL', 'HIGHEND_CD2', 'QOQ_TOT_ASST_RTC',
    'PYE_AL012G011', 'SHP_CNT', 'R3M_ITRT_SHOP_JIKGU', 'B1Y_MOB_OS',
    'HOME_ADM', 'R3M_ITRT_ENT_WEBTOON', 'R3M_ITRT_FIN_INSUR',
    'YOY_R3M_ITRT_COMM_VOIP_CS',
    # 기존 소비 변수 9개 (46개 시절부터 있던 것)
    'YOY_R3M_FOOD_AMT_RTC', 'R6M_MART_AMT', 'YOY_R3M_SSM_AMT_RTC',
    'YOY_R3M_MART_AMT_RTC', 'QOQ_R3M_MART_AMT_RTC',
    'YOY_R3M_HOUSEHOLD_AMT_RTC', 'YOY_R3M_CONV_AMT_RTC',
    'R6M_M_MART_AMT', 'YOY_R3M_M_CONV_AMT_RTC',
    # 실험 A 채택 13개
    'R6M_SSM_AMT', 'R6M_MED_AMT', 'R6M_HOUSEHOLD_AMT', 'R6M_OIL_AMT',
    'R6M_FOOD_AMT', 'R6M_CONV_AMT', 'R6M_BEAUTY_AMT', 'R6M_CUL_AMT',
    'R6M_M_CONV_AMT', 'QOQ_R3M_HOUSEHOLD_AMT_RTC', 'QOQ_R3M_E_COMM_AMT_RTC',
    'YOY_R3M_E_COMM_AMT_RTC_was_missing', 'YOY_R3M_STARBUCKS_AMT_RTC_was_missing',
    # 실험 C 채택 11개
    'CPYT_MART_AMT_RT', 'CPYT_CLOTHES_AMT_RT_was_missing', 'CPYT_HOUSEHOLD_AMT_RT',
    'CPYT_SSM_AMT_RT_was_missing', 'CPYT_M_CONV_AMT_RT', 'CPYT_OIL_AMT_RT_was_missing',
    'CPYT_SSM_AMT_RT', 'CPYT_E_COMM_AMT_RT', 'CPYT_STARBUCKS_AMT_RT_was_missing',
    'CPYT_MART_AMT_RT_was_missing', 'CPYT_STARBUCKS_AMT_RT',
    # 등급 관련 파생 5개
    'grade_unavailable', 'PET_GD_woe', 'APP_GD_woe', 'GOLF_GD_woe', 'TRAVEL_GD_woe',
    # 결측 처리 플래그 5개
    'DAR_was_missing', 'FST_CAR_ELPS_was_missing', 'FST_HOUS_BUY_ELPS_was_missing',
    'CPYT_CONV_AMT_RT_was_missing', 'CPYT_FOOD_AMT_RT_was_missing'
]
final_69_features = list(dict.fromkeys(final_69_features))
print(f"69개 확인: {len(final_69_features)}개")

# ===== 2. 원본 로드가 필요한 실제 컬럼 (파생 플래그는 원본명으로 로드 후 계산) =====
grade_source_cols = ['PET_GD', 'APP_GD', 'GOLF_GD', 'TRAVEL_GD']
thin_def_cols = ['PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004', 'PYE_C18233005', 'PYE_L10210000']

flag_derived = [c for c in final_69_features if c.endswith('_was_missing')]
raw_source_of_flags = [c.replace('_was_missing', '') for c in flag_derived]
direct_load = [c for c in final_69_features if not c.endswith('_was_missing')
               and c not in ['grade_unavailable','PET_GD_woe','APP_GD_woe','GOLF_GD_woe','TRAVEL_GD_woe']]

# ===== 3. 딱 한 번에 모든 컬럼 로드 =====
load_cols = list(dict.fromkeys(
    ['CUST_ID'] + direct_load + raw_source_of_flags + grade_source_cols + thin_def_cols
))
print(f"로드할 총 컬럼 수: {len(load_cols)}개")

df = pd.read_csv(f'{scan_path}/202103_통신카드CB결합.csv', usecols=load_cols)
gc.collect()
print(df.shape)

# ===== 4. 결측 처리 — 반복 insert 대신 pd.concat으로 한 번에 =====
missing_check = df[raw_source_of_flags].isnull().sum()
missing_cols = missing_check[missing_check > 0].index.tolist()
print(f"결측 있는 컬럼: {len(missing_cols)}개")

flag_df = df[missing_cols].isnull().astype(int)
flag_df.columns = [f'{c}_was_missing' for c in missing_cols]

df = pd.concat([df, flag_df], axis=1)
df[missing_cols] = df[missing_cols].fillna(0)
del flag_df
gc.collect()
print(f"잔여 결측: {df[raw_source_of_flags].isnull().sum().sum()}")

# ===== 5. 씬파일러 정의 + TARGET 병합 =====
thin_mask = (
    (df['PYE_C1M210000']==0) & (df['PYE_C18233003']==0) &
    (df['PYE_C18233004']==0) & (df['PYE_C18233005']==0) &
    (df['PYE_L10210000']==0)
)
df['segment'] = np.where(thin_mask, 'thin_filer', 'credit_holder')

df_y = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=['CUST_ID', 'PYE_D1011000C'])
df_y['TARGET'] = (df_y['PYE_D1011000C'] > 0).astype(int)
df = df.merge(df_y[['CUST_ID', 'TARGET']], on='CUST_ID', how='left')
del df_y
gc.collect()

credit_holders = df[df['segment']=='credit_holder'].copy()
thin_filers = df[df['segment']=='thin_filer'].copy()
del df
gc.collect()

print(f"신용이력 보유군: {len(credit_holders):,}명, 양성 {credit_holders['TARGET'].sum():,}명 ({credit_holders['TARGET'].mean()*100:.3f}%)")
print(f"씬파일러: {len(thin_filers):,}명")

# ===== 6. WOE 인코딩 (credit_holders에서만 계산, thin_filers엔 매핑만 적용) =====
credit_holders['grade_unavailable'] = (credit_holders['PET_GD'] == '*').astype(int)
thin_filers['grade_unavailable'] = (thin_filers['PET_GD'] == '*').astype(int)

def woe_encode_nonstar(df_, feature, target):
    data = df_[df_[feature] != '*'][[feature, target]].copy()
    grouped = data.groupby(feature, observed=True)[target].agg(['count', 'sum'])
    grouped.columns = ['total', 'bad']
    grouped['good'] = grouped['total'] - grouped['bad']
    total_bad = grouped['bad'].sum()
    total_good = grouped['good'].sum()
    grouped['bad_rate'] = (grouped['bad'] + 0.5) / (total_bad + 0.5)
    grouped['good_rate'] = (grouped['good'] + 0.5) / (total_good + 0.5)
    grouped['woe'] = np.log(grouped['good_rate'] / grouped['bad_rate'])
    return grouped['woe'].to_dict()

for col in grade_source_cols:
    woe_map = woe_encode_nonstar(credit_holders, col, 'TARGET')
    credit_holders[f'{col}_woe'] = credit_holders[col].map(woe_map).fillna(0)
    thin_filers[f'{col}_woe'] = thin_filers[col].map(woe_map).fillna(0)

gc.collect()
print("\nWOE 인코딩 및 69개 변수셋 준비 완료")
print(f"credit_holders 결측 확인: {credit_holders[final_69_features].isnull().sum().sum()}")

### 검증 — 69개 변수셋 XGBoost 최종 성능 확인

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import xgboost as xgb

X = credit_holders[final_69_features]
y = credit_holders['TARGET']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
    eval_metric='auc', random_state=42
)
model.fit(X_train, y_train)

pred_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, pred_proba)
pr_auc = average_precision_score(y_test, pred_proba)

print(f"[최종 69개]  Val AUC: {auc:.4f}, Val PR-AUC: {pr_auc:.4f}")
print(f"(참고 기록치: Val AUC 0.7824, Val PR-AUC 0.1820 — 시드/버전에 따라 소폭 차이 가능)")

---
## 다음 단계

1. **이 69개 변수셋을 새 기준(baseline)으로 확정** — track_b_03 산출물(train.csv/apply.csv)을 69개 기준으로 재생성 필요
2. 강사님 피드백 나머지 두 개로 이동
   - **④ 씬파일러 자체 검증**: track_b_01에서 신규대출 발생 관측(forward 구조)은 완전히 막혔음(2년 관측해도 신규대출 0명). 씬파일러 중 `PYE_D1011000C>0`인 197명(2021년에도 씬파일러였던 176명 포함)의 예측 위험점수가 나머지 씬파일러보다 높게 나오는지 확인하는 게 남은 현실적 방법
   - **⑤ 정확도 vs 정밀도**: 지금 XGBoost는 `scale_pos_weight` 적용 + PR-AUC를 핵심 지표로 쓰고 있어 사실상 정밀도 쪽에 무게. 이걸 명시적으로 정리해서 강사님께 확인 필요